# 8. Error Analysis

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from src.config.settings import load_config
from src.config.paths import (
    PROCESSED_DATA_DIR,
    ARTIFACTS_DIR,
    TUNED_MODELS_DIR
)
from src.data.load_data import load_raw_data
from src.models.training.threshold_tuning import tune_threshold, compute_threshold_curves
from sklearn.model_selection import StratifiedKFold, cross_val_predict

## 1. Load config

In [2]:
config = load_config()

TARGET = config["target"]
DROP_COLS = config["drop_columns"]
SPLIT_CONFIG = config["split"]
TUNING_CONFIG = config["tuning"]
THRESHOLD_CONFIG = config["threshold_tuning"]

print(f"Target:           {TARGET}")
print(f"Optimising for:   {THRESHOLD_CONFIG['metric']}")

Target:           good
Optimising for:   f1


## 2. Load processed data


In [3]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze()

print(f"X_train shape: {X_train.shape}")

X_train shape: (1087, 11)


## 4. Load tuned models


In [4]:
tuned_models = joblib.load(ARTIFACTS_DIR / "tuned_models.pkl")

print(f"Available tuned models: {list(tuned_models.keys())}")

model_name = TUNING_CONFIG["models_to_tune"][0]
model = tuned_models[model_name]

print(f"Tuning threshold for: {model_name}")

Available tuned models: ['xgboost_classifier', 'logistic_regression']
Tuning threshold for: xgboost_classifier


In [5]:
import os
print("cores:", os.cpu_count())
print("model n_jobs:", model.get_params().get("n_jobs"))

cores: 16
model n_jobs: None


## 5. Model Score


In [9]:
%%time
from sklearn.base import clone

model_name = TUNING_CONFIG["models_to_tune"][0]
model = tuned_models[model_name]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_cv = clone(model).set_params(n_jobs=config["n_jobs"])
y_proba_train = cross_val_predict(model_cv, X_train, y_train, cv=cv,
                                  method="predict_proba")[:, 1]

CPU times: user 6.25 s, sys: 109 ms, total: 6.36 s
Wall time: 902 ms


In [15]:
df_error = pd.DataFrame({
    "true": y_train.values,
    "proba": y_proba_train,
})

df_error["pred"] = (df_error["proba"] >= 0.5).astype(int)
df_error["wrong"] = df_error["true"] != df_error["pred"]

print(df_error["wrong"].sum())
df_error.head()


267


,true,proba,pred,wrong
0,1,0.876027,1,False
1,1,0.806481,1,False
2,0,0.563307,1,True
3,1,0.912890,1,False
4,0,0.664497,1,True


In [16]:
print(df_error["wrong"].sum(), "wrong out of", len(df_error))
print(df_error[df_error["wrong"]]["proba"].describe())

267 wrong out of 1087
count    267.000000
mean       0.498676
std        0.199228
min        0.082345
25%        0.337950
50%        0.496090
75%        0.655422
max        0.941840
Name: proba, dtype: float64


,mean,count
band,,
"(0.0, 0.2]",0.130719,153
"(0.2, 0.35]",0.257282,206
"(0.35, 0.5]",0.401235,162
"(0.5, 0.65]",0.448529,136
"(0.65, 0.8]",0.268156,179
"(0.8, 1.0]",0.079681,251


In [ ]:
from sklearn.model_selection import train_test_split
from src.data.load_data import load_raw_data, prepare_data

SPLIT_CONFIG = config["split"]

df = load_raw_data(na_values=config.get("na_values", []))
quality = df.drop_duplicates().reset_index(drop=True)["quality"]
X, y = prepare_data(df, config)

stratify_col = y if SPLIT_CONFIG["stratify"] else None

X_tr, X_te, y_tr, y_te, q_tr, q_te = train_test_split(
    X, y, quality,
    test_size=SPLIT_CONFIG["test_size"],
    random_state=SPLIT_CONFIG["random_state"],
    stratify=stratify_col,  
)

print((y_tr.values == y_train.values).all())

240 instances have been dropped
True


In [23]:
df_error["quality"] = q_tr.values
df_error.groupby("quality")["wrong"].agg(["mean", "count"])

,mean,count
quality,,
3,0.000000,7
4,0.275000,40
5,0.253763,465
6,0.303448,435
7,0.046512,129
8,0.000000,11


In [24]:
confident = df_error[df_error["wrong"] & ((df_error["proba"] > 0.8) | (df_error["proba"] < 0.2))]
print(len(confident))
confident.groupby("quality").size()

40


quality
5    20
6    20
dtype: int64

In [25]:
confident = df_error[df_error["wrong"] & ((df_error["proba"] > 0.8) | (df_error["proba"] < 0.2))]
print(len(confident))
confident.groupby("quality").size()

40


quality
5    20
6    20
dtype: int64